# The widest empty street

Many lines can separate two groups. The SVM chooses the one whose empty street is as wide as possible.

Suppose the labels are $y_i \in \{-1, +1\}$. A line separates the training points when every positive point has a positive score and every negative point has a negative score. Written as one inequality:

$$
y_i(\mathbf{w}\cdot\mathbf{x}_i+b) > 0
$$

The product $y_i(\mathbf{w}\cdot\mathbf{x}_i+b)$ is the **functional margin** of that point. The **geometric margin** of the whole set is the distance from the line to the nearest training point:

$$
\gamma = \min_i \frac{y_i(\mathbf{w}\cdot\mathbf{x}_i+b)}{\|\mathbf{w}\|}
$$


## Why the constraints are written as $\ge 1$

If $(\mathbf{w}, b)$ separates the points, so does $(c\mathbf{w}, cb)$ for every $c > 0$. The line is the same and $\gamma$ is the same. We are free to pick the scale. Choose it so the nearest point has functional margin exactly $1$:

$$
\min_i y_i(\mathbf{w}\cdot\mathbf{x}_i+b) = 1
$$

Then $\gamma = 1/\|\mathbf{w}\|$, and every point satisfies

$$
y_i(\mathbf{w}\cdot\mathbf{x}_i+b) \ge 1
$$

Maximizing the street width $2\gamma = 2/\|\mathbf{w}\|$ is the same as minimizing $\|\mathbf{w}\|$. The squared length is smoother and has the same minimizer:

$$
\begin{aligned}
\underset{\mathbf{w}, b}{\operatorname{minimize}} \quad
& \frac{1}{2}\|\mathbf{w}\|^2 \\
\text{subject to} \quad
& y_i(\mathbf{w}\cdot\mathbf{x}_i+b) \ge 1
\quad\text{for every training point } i
\end{aligned}
$$

This is the **hard-margin** primal. The factor $1/2$ does not move the answer. It cancels a $2$ when we differentiate later.


## A picture you can calculate

Positive point $(2, 0)$. Negative points $(0, 1)$ and $(0, -1)$.

The two negative points have average $(0, 0)$. The gap from that average to the positive point runs from $x_1 = 0$ to $x_1 = 2$. The widest street puts its center at $x_1 = 1$ and its edges on the data, at $x_1 = 0$ and $x_1 = 2$.

That street is exactly the score $x_1 - 1$:

$$
\mathbf{w} = (1, 0), \qquad b = -1
$$

Check the three functional margins:

$$
\begin{aligned}
(2, 0) &: (+1)(2-1) = 1 \\
(0, 1) &: (-1)(0-1) = 1 \\
(0, -1) &: (-1)(0-1) = 1
\end{aligned}
$$

All three sit on a street edge. None falls inside the street. The objective is

$$
\frac{1}{2}\|\mathbf{w}\|^2 = \frac{1}{2}
$$

and the street width is $2/\|\mathbf{w}\| = 2$.


In [1]:
# Confirm the three functional margins and the street width.
import math

w = (1, 0)
b = -1
rows = [
    ((2, 0), +1),
    ((0, 1), -1),
    ((0, -1), -1),
]
margins = []
for point, label in rows:
    value = label * (w[0] * point[0] + w[1] * point[1] + b)
    margins.append(value)
    assert value == 1
objective = 0.5 * (w[0] ** 2 + w[1] ** 2)
width = 2 / math.sqrt(w[0] ** 2 + w[1] ** 2)
assert objective == 0.5
assert width == 2
print("functional margins", margins)
print("objective", objective)
print("street width", width)


functional margins [1, 1, 1]
objective 0.5
street width 2.0


## Why a steeper line is worse

Try $\mathbf{w} = (1, 1)$ and choose $b$ so the positive point is tight. There is no need to finish a full optimization to see the cost. Even the feasible choice $\mathbf{w} = (2, 0)$, $b = -2$ separates the points, but

$$
\frac{1}{2}\|(2, 0)\|^2 = 2
$$

which is four times the objective of $(1, 0)$. Its street is only half as wide:

$$
\frac{2}{\|(2,0)\|} = 1
$$

The constraints $y_i(2x_1-2) \ge 1$ still hold, because the functional margins are $2$, $2$, and $2$. Feasible is not the same as widest.


In [2]:
# A feasible but narrower line: w = (2, 0), b = -2.
rows = [((2, 0), 1), ((0, 1), -1), ((0, -1), -1)]
for point, label in rows:
    margin = label * (2 * point[0] - 2)
    assert margin == 2
assert 0.5 * (2 ** 2) == 2
print("narrow objective", 2)
print("wide objective", 0.5)


narrow objective 2
wide objective 0.5


## A pitfall

The number $1$ on the right-hand side is a scale we chose, not a unit of the data. After that choice, $\|\mathbf{w}\|$ carries the geometry. Comparing scores from two different fits is meaningless until both fits use the same convention, or until you convert scores back to distances.

## What to carry forward

Hard-margin SVM minimizes $\tfrac{1}{2}\|\mathbf{w}\|^2$ while keeping every functional margin at least $1$. That is the widest street that does not touch a training point.
